# question 1

In [132]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import mode

col_names = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df = pd.read_csv('car_evaluation.csv', header=None, names=col_names)

print("First 5 rows of the dataset:")
display(df.head())

X = df.drop(['class'], axis=1)
y = df['class']

ordinal_encoder = OrdinalEncoder()
X_encoded = ordinal_encoder.fit_transform(X)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
class MyBaggingClassifier:
    def __init__(self, base_estimator, n_estimators=10):
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.estimators_ = []

    def fit(self, X, y):
        n_samples = X.shape[0]
        
        for _ in range(self.n_estimators):
            # Create a bootstrap sample (sampling with replacement)
            indices = np.random.choice(n_samples, size=n_samples, replace=True)
            X_boot, y_boot = X[indices], y[indices]
            
            # Train a new base estimator on the bootstrap sample
            estimator = self.base_estimator
            estimator.fit(X_boot, y_boot)
            self.estimators_.append(estimator)
            
        return self

    def predict(self, X):
        predictions = np.array([estimator.predict(X) for estimator in self.estimators_])
        
        majority_vote, _ = mode(predictions, axis=0, keepdims=False)
        0
        return majority_vote

print("--- Single Decision Tree ---")

single_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
single_tree.fit(X_train, y_train)
y_pred_tree = single_tree.predict(X_test)
accuracy_tree = accuracy_score(y_test, y_pred_tree)
print(f"Single Decision Tree Accuracy: {accuracy_tree:.4f}")

print("\n--- Custom Bagging Model ---")

base_estimator_bagging = DecisionTreeClassifier(max_depth=3, random_state=42)

my_bagging_model = MyBaggingClassifier(base_estimator=base_estimator_bagging, n_estimators=10)
my_bagging_model.fit(X_train, y_train)

y_pred_bagging = my_bagging_model.predict(X_test)
accuracy_bagging = accuracy_score(y_test, y_pred_bagging)
print(f"Custom Bagging Model Accuracy: {accuracy_bagging:.4f}")

First 5 rows of the dataset:


,buying,maint,doors,persons,lug_boot,safety,class
0,buying price,maintenance cost,number of doors,number of persons,lug_boot,safety,decision
1,vhigh,vhigh,2,2,small,low,unacc
2,vhigh,vhigh,2,2,small,med,unacc
3,vhigh,vhigh,2,2,small,high,unacc
4,vhigh,vhigh,2,2,med,low,unacc


Training set shape: (1383, 6)
Testing set shape: (346, 6)
--- Single Decision Tree ---
Single Decision Tree Accuracy: 0.7746

--- Custom Bagging Model ---
Custom Bagging Model Accuracy: 0.7457


# discussion

Instead of relying on one decision tree, what if we trained many trees on slightly different versions of the data and then let them vote on the final answer? This is Bagging. It's like asking a diverse committee of experts for their opinion instead of just one. This helps to reduce errors and makes the model more stable.                                                                                                                     Accuracy of a Single Decision Tree <= Accuracy of our Custom Bagging Model

# question 2

In [133]:
class MyAdaBoostClassifier:
    def __init__(self, base_estimator, n_estimators=10):
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.estimators_ = []
        self.alphas_ = [] # Stores the "amount of say" for each estimator
        self.n_classes_ = 0

    def fit(self, X, y):
        n_samples = X.shape[0]
        self.n_classes_ = len(np.unique(y))
    
        w = np.full(n_samples, (1 / n_samples))
        
        for _ in range(self.n_estimators):
            estimator = self.base_estimator
            estimator.fit(X, y, sample_weight=w)
            
            y_pred = estimator.predict(X)
            misclassified = (y_pred != y)
            
            err = np.sum(w[misclassified])
            
            if err == 0:
                self.estimators_.append(estimator)
                self.alphas_.append(1.0)
                break
            
            alpha = np.log((1.0 - err) / err) + np.log(self.n_classes_ - 1)
            
            w[misclassified] *= np.exp(alpha)
            
            w /= np.sum(w)
            
            self.estimators_.append(estimator)
            self.alphas_.append(alpha)
            
        return self

    def predict(self, X):
        class_votes = np.zeros((X.shape[0], self.n_classes_))
        
        for alpha, estimator in zip(self.alphas_, self.estimators_):
            predictions = estimator.predict(X)
            for i, pred_class in enumerate(predictions):
                class_votes[i, pred_class] += alpha
        
        return np.argmax(class_votes, axis=1)

print("--- Custom AdaBoost Model ---")

base_estimator_boosting = DecisionTreeClassifier(max_depth=1, random_state=42)

my_adaboost_model = MyAdaBoostClassifier(base_estimator=base_estimator_boosting, n_estimators=10)
my_adaboost_model.fit(X_train, y_train)

y_pred_adaboost = my_adaboost_model.predict(X_test)
accuracy_adaboost = accuracy_score(y_test, y_pred_adaboost)
print(f"Custom AdaBoost Model Accuracy: {accuracy_adaboost:.4f}")


--- Custom AdaBoost Model ---
Custom AdaBoost Model Accuracy: 0.2746


# discussion

Why wasn't AdaBoost the best here? AdaBoost is very powerful, but it didn't shine in this particular case. Here's why:

It learns sequentially: Bagging trains all its trees at once, independently. AdaBoost trains its models one by one. The first model makes some mistakes. The second model is specifically trained to fix those mistakes. The third model focuses on the mistakes made by the first two, and so on. It's like building a team of specialists.

Focus on mistakes can be risky: If the data has "noisy" or very tricky samples that are hard to classify, AdaBoost might try too hard to get them right and end up messing up the bigger picture. In our case, the 10 simple "stumps" might have struggled too much with the complexity of the car data, leading to a performance that wasn't better than a single, slightly deeper tree.